In [0]:
   %run ./01_common

In [0]:
dbutils.widgets.text("base_path", "/Volumes/workspace/github_health/raw/bronze")
dbutils.widgets.text("run_date", "*")               # "2026-10-07" for one day, "*" for everything
dbutils.widgets.text("entities", "commits,issues,pulls,contributors,repo_metadata")
dbutils.widgets.text("repo_glob", "*")              # e.g. apache__spark
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"])

def raw_to_bronze(entity, base_path, run_date, repo_glob, load_type, batch_id):
    start = datetime.datetime.now()
    path = f"{base_path}/{entity}/{repo_glob}/{entity}_{run_date}*.jsonl"
    schema, pk = SCHEMAS[entity], BRONZE_PK[entity]
    try:
        raw = spark.read.text(path).select(F.col("value").alias("_raw_json"),
                                           F.col("_metadata.file_path").alias("source_file"))
        parsed = (raw
            .withColumn("d", F.from_json("_raw_json", schema, {"mode": "PERMISSIVE"}))
            .withColumn("_keys", F.map_keys(F.from_json("_raw_json", "map<string,string>"))))
        n_read = parsed.count()

        # schema drift: JSON keys that are not in our schema
        known = [f.name for f in schema.fields]
        drift_df = parsed.select(F.explode(F.array_except("_keys", F.array(*[F.lit(c) for c in known]))).alias("c")).distinct()
        new_cols = [r.c for r in drift_df.collect()]

        valid = F.col("d").isNotNull()
        for k in pk:
            valid &= F.col(f"d.{k}").isNotNull()

        good = (parsed.filter(valid).select("d.*", "_raw_json", "source_file")
                .withColumn("batch_id", F.lit(batch_id))
                .withColumn("load_timestamp", F.current_timestamp()))
        w = Window.partitionBy(*pk).orderBy(F.col("_ingested_at").desc())
        good = good.withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")

        bad = (parsed.filter(~valid)
               .select(F.lit(entity).alias("entity"), "source_file", "_raw_json",
                       F.lit("null primary key / type mismatch / corrupt JSON").alias("reason"),
                       F.lit(batch_id).alias("batch_id"),
                       F.sha2("_raw_json", 256).alias("record_hash"),
                       F.current_timestamp().alias("load_timestamp")))
        n_bad = bad.count()
        if n_bad:
            upsert(bad, T("bronze_quarantine"), ["record_hash", "entity"])

        ins, upd = upsert(good, T(f"bronze_{entity}"), pk, "t._raw_json <> s._raw_json")
        log_run(batch_id, "Raw-to-Bronze", entity, path, load_type, start, "SUCCESS",
                n_read, ins, upd, n_bad, ",".join(new_cols) or None)
    except Exception as e:
        status = "SKIPPED" if "PATH_NOT_FOUND" in str(e) else "FAILURE"
        log_run(batch_id, "Raw-to-Bronze", entity, path, load_type, start, status, err=str(e)[:2000])

batch_id = str(uuid.uuid4())
for e in dbutils.widgets.get("entities").split(","):
    raw_to_bronze(e.strip(), dbutils.widgets.get("base_path"), dbutils.widgets.get("run_date"),
                  dbutils.widgets.get("repo_glob"), dbutils.widgets.get("load_type"), batch_id)
display(spark.table(T("pipeline_execution_logs")).orderBy(F.desc("start_time")).limit(20))